# ECG pipeline notebook – bővített 1ch / 2ch / 12ch workflow

Ez a notebook a teljes EKG-feldolgozó workflow-t **közvetlen függvényhívásokkal** futtatja.  
A fő újdonság, hogy a korábbi 1 csatornás baseline-ok mellett bekerült:

- **MIT-BIH 2 csatornás CNN**
- **INCART 12 csatornás CNN**
- bővített összehasonlítás a compare szekcióban

**Rövid értelmezés**
- A **feature extraction**, a **morphology** és a **QC** továbbra is 1 csatornás baseline-on fut.
- A **CNN** viszont már tud külön 1ch / 2ch / 12ch változatban tanulni.
- Így a klasszikus és a deep learning alapú megközelítés tisztán összehasonlítható.

**Használat**
- Egy szekció újrafuttatásához: `reset_section("mitbih")`
- Egy adott lépés automatikusan skipelődik, ha már completed
- Kényszerített újrafuttatáshoz a notebook util logikát használd vagy töröld az adott szekció checkpointját

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

In [ ]:
# -------------------------
# IMPORTOK – core függvények
# -------------------------

from src.notebook_utils import run_step_if_needed, reset_section, checkpoint_path, load_checkpoint
from src.config import OUTPUT_DIR

from src.download_mitbih_paralel import download_mitbih_paralel
from src.download_incart_paralel import download_incart_paralel

from src.qc_dataset_ext import qc_dataset_ext
from src.plot_preprocessing_comparison import plot_preprocessing_comparison

from src.prepare_mitbih_datasets import prepare_mitbih_datasets
from src.prepare_mitbih_datasets_2ch import prepare_mitbih_datasets_2ch
from src.prepare_incart_datasets import prepare_incart_datasets
from src.prepare_incart_datasets_12ch import prepare_incart_datasets_12ch

from src.build_general_features import build_general_features
from src.build_morphology_features import build_morphology_features
from src.report_features import report_features
from src.train_featurext import train_featurext
from src.plot_dataset_tsne import plot_dataset_tsne
from src.plot_tsne_featurext_vs_cnn import plot_tsne_featurext_vs_cnn
from src.train_cnn import train_cnn

from src.prepare_cross_dataset_splits import prepare_cross_dataset_splits
from src.plot_cross_dataset_tsne import plot_cross_dataset_tsne
from src.compare_models_extended import compare_dataset_models_extended

## MIT-BIH

Ebben a szekcióban először lefut a **szokásos 1 csatornás baseline** teljes feldolgozása, majd külön elkészül a **2 csatornás CNN input** és a hozzá tartozó tanítás.  
Így a compare résznél közvetlenül összehasonlítható lesz:

- FeatureXt
- CNN 1ch
- CNN 2ch

In [ ]:
SECTION = "mitbih"
COMMON = {"dataset": "mitbih"}

run_step_if_needed(SECTION, download_mitbih_paralel)
run_step_if_needed(SECTION, qc_dataset_ext, COMMON)
run_step_if_needed(SECTION, plot_preprocessing_comparison, {**COMMON, "record_name": "100"})

# 1ch baseline prepare + feature pipeline
run_step_if_needed(SECTION, prepare_mitbih_datasets)
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)
run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, COMMON)

# 2ch CNN ág
run_step_if_needed(SECTION, prepare_mitbih_datasets_2ch, step_name="prepare_mitbih_datasets_2ch")
run_step_if_needed(
    SECTION,
    train_cnn,
    {**COMMON, "variant": "2ch"},
    step_name="train_cnn_2ch",
)

## INCART

Az INCART-nál a baseline 1 csatornás pipeline megmarad, de külön elkészül a **12 csatornás CNN dataset** is.  
Ez azért hasznos, mert itt közvetlenül mérhető, mennyit ad hozzá a több elvezetés a klasszifikációhoz.

In [ ]:
SECTION = "incart"
COMMON = {"dataset": "incart"}

run_step_if_needed(SECTION, download_incart_paralel)
run_step_if_needed(SECTION, qc_dataset_ext, COMMON)
run_step_if_needed(SECTION, plot_preprocessing_comparison, {**COMMON, "record_name": "I01"})

# 1ch baseline prepare + feature pipeline
run_step_if_needed(SECTION, prepare_incart_datasets)
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)
run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, COMMON)

# 12ch CNN ág
run_step_if_needed(SECTION, prepare_incart_datasets_12ch, step_name="prepare_incart_datasets_12ch")
run_step_if_needed(
    SECTION,
    train_cnn,
    {**COMMON, "variant": "12ch"},
    step_name="train_cnn_12ch",
)

## CROSS TEST

Ez a szekció a cross-dataset általánosítási kísérletekhez tartozik.  
A jelen notebookban ez a rész **az eredeti baseline logikát követi**, vagyis itt elsősorban az 1 csatornás összehasonlítás a fő fókusz.

In [ ]:
SECTION = "cross_test"
BASE = OUTPUT_DIR / "cross_dataset" / "cross_test"

run_step_if_needed(SECTION, prepare_cross_dataset_splits)

run_step_if_needed(
    SECTION,
    train_cnn,
    {
        "dataset": "cross_test",
        "data_dir": str(BASE),
        "results_dir": str(OUTPUT_DIR / "cross_test_cnn"),
    },
)

# A feature-es és morphology-s cross-dataset ág itt meghagyható külön,
# ha a saját projektverziódban ezek a wrapper-ek támogatják a szükséges paramétereket.

## MIXED

Vegyes tanítóhalmaz: MIT-BIH + INCART.  
Ez a rész továbbra is főleg arra jó, hogy lásd, mennyit segít a vegyes tanítás a generalizáción.

In [ ]:
SECTION = "mixed"
BASE = OUTPUT_DIR / "cross_dataset" / "mixed"

run_step_if_needed(SECTION, prepare_cross_dataset_splits)

run_step_if_needed(
    SECTION,
    train_cnn,
    {
        "dataset": "mixed",
        "data_dir": str(BASE),
        "results_dir": str(OUTPUT_DIR / "mixed_cnn"),
    },
)

## DOMAIN GENERALIZATION

Ebben a blokkban a cél az, hogy a tanítás és a teszt különböző domainből jöjjön, és látszódjon, mennyire robusztus a modell.  
Itt is a baseline CNN ág marad a fő futási út.

In [ ]:
SECTION = "domain_generalization"
BASE = OUTPUT_DIR / "cross_dataset" / "domain_generalization"

run_step_if_needed(SECTION, prepare_cross_dataset_splits)

run_step_if_needed(
    SECTION,
    train_cnn,
    {
        "dataset": "domain_generalization",
        "data_dir": str(BASE),
        "results_dir": str(OUTPUT_DIR / "domain_generalization_cnn"),
    },
)

## COMPARE

Ebben a részben készülnek az összehasonlító ábrák és táblázatok.

**Rövid értelmezés**
- MIT-BIH-nál a fő kérdés: javít-e a **2 csatornás CNN** az 1 csatornás baseline-hoz képest.
- INCART-nál a fő kérdés: mennyit ad hozzá a **12 csatorna** a FeatureXt és az 1ch CNN mellé.
- A compare script már a confusion matrix-eket is egymás mellé tudja rajzolni.

In [ ]:
SECTION = "compare"

run_step_if_needed(
    SECTION,
    plot_cross_dataset_tsne,
    {
        "source_dataset": "mitbih",
        "target_dataset": "incart",
        "split": "test",
    },
)

run_step_if_needed(
    SECTION,
    compare_dataset_models_extended,
    {
        "dataset": "mitbih",
        "left_family": "featurext",
        "right_family": "cnn",
        "third_family": "cnn2",
        "results_subdir": "mitbih_featurext_vs_cnn_vs_cnn2_extended",
        "select_best_left": True,
    },
    step_name="compare_models_extended_mitbih_3way",
)

run_step_if_needed(
    SECTION,
    compare_dataset_models_extended,
    {
        "dataset": "incart",
        "left_family": "featurext",
        "right_family": "cnn",
        "third_family": "cnn12",
        "results_subdir": "incart_featurext_vs_cnn_vs_cnn12_extended",
        "select_best_left": True,
    },
    step_name="compare_models_extended_incart_3way",
)

run_step_if_needed(
    SECTION,
    compare_dataset_models_extended,
    {
        "dataset": "incart",
        "left_family": "cnn",
        "right_family": "cnn12",
        "results_subdir": "incart_cnn_vs_cnn12ch_extended",
    },
    step_name="compare_models_extended_incart_cnn_vs_cnn12",
)

## Rövid záró értelmezés

A notebook végére várhatóan három fontos összehasonlításod lesz:

1. **MIT-BIH:** FeatureXt vs CNN 1ch vs CNN 2ch  
2. **INCART:** FeatureXt vs CNN 1ch vs CNN 12ch  
3. **Cross-dataset:** mennyire romlik a teljesítmény domainváltásnál

Érdemes külön figyelni:
- a **macro-F1** változására,
- a **confusion matrix**-ben a V és OTHER osztályok keveredésére,
- valamint arra, hogy a többcsatornás modellek valóban stabilabban viselkednek-e a teszt splitben.